# GestuRhythm v2 - Music Prior Training
Train GPT-style model hoc ngu phap am nhac tu Lakh MIDI dataset.
Model hoc P(note_t | note_0..t-1) â€” hoan toan doc lap voi cu chi.
Output: music_prior.pt

In [ ]:
import os, glob, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Cai pretty_midi neu chua co
os.system('pip install pretty_midi -q')
import pretty_midi
print('pretty_midi OK')

In [ ]:
# Debug: tim duong dan dung cua dataset
import os, glob
print('Cac dataset kha dung:')
for d in sorted(os.listdir('/kaggle/input/')):
    p = f'/kaggle/input/{d}/'
    sub = os.listdir(p)[:3]
    print(f'  {p} -> {sub}')

# Tim file mid bat ky
all_mid = glob.glob('/kaggle/input/**/*.mid', recursive=True)
all_mid += glob.glob('/kaggle/input/**/*.MID', recursive=True)
print(f'Tim thay {len(all_mid)} file .mid')
if all_mid:
    print('Vi du:', all_mid[0])


In [ ]:
# Tai Lakh MIDI (clean subset)
# Su dung Lakh MIDI Dataset tren Kaggle: 'imsparsh/lakh-midi-dataset'
# Dataset chia theo thu muc nghe si -> can tim recursive
MIDI_DIR = '/kaggle/input/lakh-midi-clean/'
midi_files = []
for root, dirs, files in os.walk(MIDI_DIR):
    for f in files:
        if f.lower().endswith(('.mid', '.midi')):
            midi_files.append(os.path.join(root, f))
print(f'Tim thay {len(midi_files)} file MIDI')
random.shuffle(midi_files)
midi_files = midi_files[:5000]
print(f'Dung {len(midi_files)} file')

In [ ]:
# Trich xuat melody tokens tu MIDI
# Token: 0-127=MIDI note, 128=REST, 129=PAD
REST = 128
PAD  = 129
SEQ_LEN = 128

def midi_to_tokens(midi_path, max_notes=256):
    try:
        pm = pretty_midi.PrettyMIDI(midi_path)
        # Lay track co nhieu note nhat (melody)
        best = max(pm.instruments, key=lambda x: len(x.notes), default=None)
        if best is None or len(best.notes) < 8:
            return None
        # Sort theo thoi gian
        notes = sorted(best.notes, key=lambda n: n.start)[:max_notes]
        tokens = []
        prev_end = 0.0
        for note in notes:
            # Them REST neu co khoang trong > 0.2s
            if note.start - prev_end > 0.2:
                tokens.append(REST)
            tokens.append(int(np.clip(note.pitch, 0, 127)))
            prev_end = note.end
        return tokens if len(tokens) >= 16 else None
    except Exception:
        return None

print('Trich xuat tokens...')
all_tokens = []
for i, f in enumerate(midi_files):
    t = midi_to_tokens(f)
    if t: all_tokens.append(t)
    if (i+1) % 500 == 0:
        print(f'  {i+1}/{len(midi_files)} | {len(all_tokens)} sequences ok')

print(f'\nTong: {len(all_tokens)} sequences hop le')
avg_len = np.mean([len(t) for t in all_tokens])
print(f'Do dai trung binh: {avg_len:.0f} tokens')

In [ ]:
class MusicPriorDataset(Dataset):
    def __init__(self, token_seqs, seq_len=SEQ_LEN):
        self.data, self.seq_len = [], seq_len
        for seq in token_seqs:
            # Sliding window
            for start in range(0, len(seq) - seq_len, seq_len // 2):
                chunk = seq[start:start + seq_len + 1]
                if len(chunk) == seq_len + 1:
                    self.data.append(chunk)
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        t = self.data[i]
        return torch.tensor(t[:-1], dtype=torch.long), torch.tensor(t[1:], dtype=torch.long)

random.shuffle(all_tokens)
n_train = int(0.9 * len(all_tokens))
train_ds = MusicPriorDataset(all_tokens[:n_train])
val_ds   = MusicPriorDataset(all_tokens[n_train:])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
class MusicPrior(nn.Module):
    VOCAB = 130
    def __init__(self, vocab_size=130, d_model=128, nhead=4, num_layers=4, max_len=512):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model, padding_idx=129)
        self.pos_enc = nn.Embedding(max_len, d_model)
        enc_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=256,
            dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head        = nn.Linear(d_model, vocab_size)

    def forward(self, tokens):
        B, T = tokens.shape
        mask = torch.triu(torch.ones(T, T, device=tokens.device), diagonal=1).bool()
        pos  = torch.arange(T, device=tokens.device).unsqueeze(0)
        x    = self.embed(tokens) + self.pos_enc(pos)
        return self.head(self.transformer(x, mask=mask))

    def get_hidden(self, tokens):
        B, T = tokens.shape
        mask = torch.triu(torch.ones(T, T, device=tokens.device), diagonal=1).bool()
        pos  = torch.arange(T, device=tokens.device).unsqueeze(0)
        x    = self.embed(tokens) + self.pos_enc(pos)
        return self.transformer(x, mask=mask)  # (B, T, d_model)

    @torch.no_grad()
    def generate(self, primer, max_len=64, temperature=1.0):
        self.eval()
        tokens = primer.clone()
        for _ in range(max_len - tokens.shape[1]):
            logits = self.forward(tokens)[:, -1, :] / temperature
            next_t = torch.multinomial(F.softmax(logits, dim=-1), 1)
            tokens = torch.cat([tokens, next_t], dim=1)
        return tokens

model = MusicPrior().to(device)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
criterion = nn.CrossEntropyLoss(ignore_index=129)

EPOCHS = 30
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    tl = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)  # (B, T, vocab)
        loss   = criterion(logits.reshape(-1, 130), yb.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tl += loss.item()
    model.eval()
    vl = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            vl += criterion(logits.reshape(-1, 130), yb.reshape(-1)).item()
    scheduler.step()
    train_losses.append(tl / len(train_loader))
    val_losses.append(vl / len(val_loader))
    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1:2d}/{EPOCHS} | train={train_losses[-1]:.4f} | val={val_losses[-1]:.4f}')
print('Done!')

In [ ]:
# Bieu do loss
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Music Prior Training', fontweight='bold')

axes[0].plot(train_losses, label='Train', color='steelblue')
axes[0].plot(val_losses,   label='Val',   color='coral')
axes[0].set_title('Loss (CrossEntropy)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Sample generation
primer  = torch.tensor([[60, 64, 67]], device=device)  # C major chord
gen     = model.generate(primer, max_len=32, temperature=0.8)[0].cpu().numpy()
note_names = ['C','C#','D','Eb','E','F','F#','G','Ab','A','Bb','B']
gen_names  = [note_names[n%12] if n < 128 else 'REST' for n in gen]
axes[1].bar(range(len(gen)), [n if n < 128 else 0 for n in gen], color='mediumpurple')
axes[1].set_title('Generated Melody (primer: C-E-G)')
axes[1].set_xlabel('Note index'); axes[1].set_ylabel('MIDI pitch')
print('Generated:', ' '.join(gen_names))

plt.tight_layout()
plt.savefig('music_prior_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
torch.save({
    'model_state': model.state_dict(),
    'config': {'vocab_size':130,'d_model':128,'nhead':4,'num_layers':4,'max_len':512},
    'val_loss': val_losses[-1],
}, 'music_prior.pt')
print('Saved: music_prior.pt')
print(f'Final val loss: {val_losses[-1]:.4f}')